In [ ]:
# pip install -U ddgs
from typing import Annotated

from typing_extensions import TypedDict
from langchain_community.tools.ddg_search import DuckDuckGoSearchRun

from kagraph import END, StateGraph, START, add_messages, invoke_llm
from kagraph.llms import load_llm
from kagraph.messages import AnyMessage, HumanMessage, SystemMessage
from kagraph.prebuilt import ToolNode, tools_condition
from kagraph.tracing import trace

In [ ]:
# ==============================================================================
# Setup Tools & LLM
# ==============================================================================
web_search_tool = DuckDuckGoSearchRun()

def web_search(query: str) -> str:
    """Search the web for information using DuckDuckGo."""
    return web_search_tool.invoke(query)

tools = [web_search]

In [ ]:
# Using KaGraph's native LLM loader
llm = load_llm("qwen/qwen3-235b-a22b-instruct-2507")

In [ ]:
# ==============================================================================
# Graph State and Nodes
# ==============================================================================
class ReActInput(TypedDict, total=False):
    question: str

class GraphState(TypedDict, total=False):
    """
    Represents the state of our graph.
    """
    messages: Annotated[list[AnyMessage], add_messages]

def prepare_question(state: ReActInput):
    print("---PREPARE QUESTION---")
    return {"messages": [HumanMessage(state["question"])]}

def agent(state: GraphState):
    print("---AGENT---")

    sys_msg = SystemMessage(
        "You are a helpful AI assistant with access to web search. "
        "Use the tools provided to answer the user's questions."
    )
    
    # Use invoke_llm for full messaging/tool-calling context tracing
    response = invoke_llm(llm, prompt=None, messages=state["messages"], system=sys_msg.content, tools=tools)
    return {"messages": [response]}

In [ ]:
# ==============================================================================
# Compile Graph
# ==============================================================================
workflow = StateGraph(GraphState, input_schema=ReActInput)

workflow.add_node("prepare_question", prepare_question, input_schema=ReActInput)
workflow.add_node("agent", agent)
workflow.add_node("tools", ToolNode(tools))

workflow.add_edge(START, "prepare_question")
workflow.add_edge("prepare_question", "agent")

# tools_condition will check if the last message has tool calls
workflow.add_conditional_edges(
    "agent",
    tools_condition,
    {
        "tools": "tools",
        END: END,
    },
)

# Loop back to agent after tool execution
workflow.add_edge("tools", "agent")

app = workflow.compile()

In [ ]:
app.get_graph().draw_png()

In [ ]:
with trace("ReAct"):
    inputs = {
        "question": "Top 3 sights to see in Vietnam? Please use tool call"
    }
    result = app.invoke(inputs)
print("FINAL ANSWER:", result["messages"][-1].content)